# Tunable PDH tones and photon counting
Use the matching PDH3 bitstream/HWH and `pdh_tones.py`. Run your normal reference-clock startup first. Loading a bitstream resets all loops; attaching with `download=False` does not. This notebook does not change intensity or PLL controllers.

In [ ]:
import importlib, time
from pathlib import Path
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import xrfdc  # register RFDC driver before Overlay
from pynq import Overlay, allocate
import pdh_tones as pdh
importlib.invalidate_caches()
pdh = importlib.reload(pdh)
# Set this to the matching file copied by your packaging monitor.
bitfile = "/home/xilinx/Bitfiles/FreqLocksBranch/overlays/TransportPhaseLockFF.bit"
# To load the new image after normal clock startup (resets existing loops):
# ol = Overlay(bitfile)
# Otherwise attach to the already loaded matching image; no clock restart/download.
ol = Overlay(bitfile, download=False)
pdh.check(ol)
print(pdh.HELPER_VERSION, pdh.status(ol))

## PDH outputs
DAC230/0 supplies the summed carrier + 60 MHz drive. DAC228/2 supplies the coherent 60 MHz reference. `configure()` mutes these outputs and sets their RFDC mixers to bypass. Other DACs are not configured by this helper. Amplitudes are fractions of digital full scale; confirm physical RF power at your EOM.

In [ ]:
print(pdh.configure(ol))  # muted; reset frequencies are 1.2 GHz and 60 MHz
print(pdh.set_levels(ol, carrier_amplitude=0.55,
                     reference_amplitude=0.10, reference_phase_deg=150))

### Frequency commands and phase continuity
A 48-bit FCW means turns per 3.44064-GS/s sample for both oscillators. The active FCW slews by at most 1024 counts/clock (3.076 MHz/s). A 1 MHz move takes 0.325 s; 100 MHz takes 32.5 s. Frequency readbacks are targets until the ramp flags clear. Neither a frequency write nor muting resets phase.

In [ ]:
carrier_hz = 1_200_001_000.0
fcw = pdh.frequency_word(carrier_hz)
print("48-bit FCW:", fcw, hex(fcw))
print(pdh.set_fcw(ol, carrier_fcw=fcw))
# Equivalent: pdh.set_frequency(ol, carrier_hz=carrier_hz)
print(pdh.wait_frequency(ol))
pdh.set_reference_phase(ol, 330)  # 180° from the previous phase: invert reference
# pdh.stop(ol)

## Photon wiring and histogram setup
Photon pulses: package **M14** (`pmod_photon_counter_pulses`). Trigger: **N14** (`pmod_photon_counter_trigger`). Both are **1.8 V logic**. Connect a common ground and use level translation if required. Photon pulses are 9.9 ns wide, up to 37.5 MHz, with detector dead time 23.3 ns. Trigger should be ≥30 ns high; 100 ns is convenient.

This is a time-binned photon count after one external trigger. The trigger is synchronized and debounced for five clocks; time zero below is that accepted trigger, with a fixed offset from the connector. Bin-boundary uncertainty is approximately one 4.069 ns fabric clock.

Choose a bin duration and total acquisition time. The 4096-word FIFO absorbs brief stalls, but very small bins can exceed DMA throughput. Start at 1 µs bins. The DMA supports at most 262143 32-bit bins per transfer.

In [ ]:
clock_hz = 245.76e6
requested_bin_s = 1e-6
number_of_bins = 100_000
cycles_per_bin = max(1, round(requested_bin_s * clock_hz))
assert 1 <= cycles_per_bin < 2**31
assert 1 <= number_of_bins <= 262143
bin_s = cycles_per_bin / clock_hz
counter = ol.photon_counting
counter.axi_gpio_0.write(0, number_of_bins)
counter.axi_gpio_0.write(8, cycles_per_bin)
time.sleep(0.001)  # allow GPIO handshake to finish before triggering
print(f"Bin: {bin_s*1e6:.6f} us; record: {number_of_bins*bin_s:.6f} s")

### Arm, then apply the external trigger
Run the next cell first. It prints `Armed` and waits; then issue the physical trigger. It times out after 10 seconds if no shot completes. Do not reprogram or change histogram settings while a capture is running. The final high-bit marker means FIFO overflow: discard the record and use longer bins.

Interrupting stops this receive channel. Before rearming after interruption, stop external triggers and let any active histogram finish; otherwise a partial previous shot can remain in the FIFO. Reloading the bitstream clears it but also resets every other loop.

In [ ]:
dma = counter.axi_dma_3.recvchannel
with allocate(shape=(number_of_bins,), dtype=np.uint32) as buffer:
    dma.transfer(buffer)
    print("Armed: apply the external trigger now.", flush=True)
    try:
        deadline = time.monotonic() + 10 + number_of_bins*bin_s
        while not dma.idle:
            if dma.error:
                raise RuntimeError("Photon DMA error; inspect its register_map.")
            if time.monotonic() > deadline:
                raise TimeoutError("No complete photon record; check trigger and DMA status.")
            time.sleep(.005)
        dma.wait()
        counts = np.asarray(buffer).copy()
        received_bytes = dma.transferred
    except BaseException:
        dma.stop()
        raise
assert not np.any(counts[:received_bytes//4] & 0x80000000), "Invalid histogram: FIFO overrun"
assert received_bytes == 4*number_of_bins, f"Incomplete histogram: {received_bytes} bytes"
time_s = (np.arange(number_of_bins)+.5)*bin_s
output_dir = Path("photon_histograms")
output_dir.mkdir(exist_ok=True)
filename = output_dir / f"photon_{datetime.now():%Y%m%d_%H%M%S}.npz"
np.savez_compressed(filename, counts=counts, time_s=time_s, bin_s=bin_s,
                    clock_hz=clock_hz, cycles_per_bin=cycles_per_bin)
print("Saved:", filename.resolve(), "total photons:", int(counts.sum()))
plt.plot(time_s*1e3, counts/bin_s/1e6)
plt.xlabel("Time after accepted trigger [ms]")
plt.ylabel("Count rate [MHz]")
plt.grid(); plt.show()